# TrackItRight — Client Message Classifier
### Training and evaluating a custom text-classification model as a comparison to the LLM extraction pipeline

TrackItRight's main app classifies client messages into `deadline`, `task`, or `extra_work` using a large language model (Groq/Llama), prompted at run time. This notebook explores an alternative approach: **training a small, dedicated classifier from scratch** on a labeled dataset, and comparing its performance and tradeoffs against the LLM approach.

**Why do this at all, if the LLM already works?**
- It demonstrates the difference between *prompting* a general-purpose model vs. *training* a task-specific one.
- A trained model runs instantly, with no API call, no cost per request, and no dependency on a third-party service being online.
- It gives a concrete way to measure accuracy on this specific task, rather than trusting the LLM's output on faith.

**Dataset:** Since no existing public dataset matches this exact classification scheme, a template-based synthetic dataset of 144 realistic client messages was constructed, covering six industries (law, architecture, interior design, marketing, software development, event planning), evenly split across the three labels.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

sns.set_style("whitegrid")

## 2. Load and Explore the Data

In [ ]:
df = pd.read_csv('client_messages_dataset.csv')
print("Dataset shape:", df.shape)
df.head(10)

In [ ]:
print(df['label'].value_counts())
print()
print(df['field'].value_counts())

The dataset is perfectly balanced across all three classes, and spread evenly across six industries — this prevents the model from simply learning to predict whichever class is most common.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#C9A227', '#5B7B8C', '#A6543A'])
axes[0].set_title('Messages per Label')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)

df['text'].str.len().plot(kind='hist', bins=20, ax=axes[1], color='#1F3D2B')
axes[1].set_title('Message Length Distribution (characters)')

plt.tight_layout()
plt.show()

## 3. Train / Test Split

The data is split 80/20, using `stratify` so the train and test sets each keep the same proportion of the three classes — important on a small dataset like this, to avoid one split accidentally being unbalanced.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f"Training examples: {len(X_train)}")
print(f"Test examples: {len(X_test)}")

## 4. Turning Text into Numbers (TF-IDF)

Machine learning models work with numbers, not raw sentences. TF-IDF (Term Frequency–Inverse Document Frequency) converts each message into a vector of numbers, where common, uninformative words (like "the," "and") are weighted down, and words that are more distinctive to a particular message are weighted up. `ngram_range=(1,2)` means it looks at both single words and pairs of consecutive words (e.g. "extra work"), which helps capture short phrases that matter for this task.

In [ ]:
vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), stop_words='english')

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Training matrix shape:", X_train_vec.shape)

## 5. Training Two Baseline Models

Two classic, lightweight text-classification models are trained and compared:
- **Logistic Regression** — a linear model, often a strong baseline for text classification.
- **Multinomial Naive Bayes** — a probabilistic model that's traditionally very effective for text, especially on smaller datasets.

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_vec, y_train)
lr_preds = lr.predict(X_test_vec)

nb = MultinomialNB()
nb.fit(X_train_vec, y_train)
nb_preds = nb.predict(X_test_vec)

print("Logistic Regression accuracy:", accuracy_score(y_test, lr_preds))
print("Naive Bayes accuracy:        ", accuracy_score(y_test, nb_preds))

## 6. Detailed Evaluation

In [ ]:
print("=== Logistic Regression ===")
print(classification_report(y_test, lr_preds))

print("=== Naive Bayes ===")
print(classification_report(y_test, nb_preds))

**Reading these metrics:**
- **Precision** — of everything the model labeled as, say, `extra_work`, what fraction actually was?
- **Recall** — of everything that actually was `extra_work`, what fraction did the model correctly catch?
- **F1-score** — the balance between precision and recall.

For TrackItRight's use case, **recall on `extra_work` matters most** — missing a genuine piece of billable extra work (a false negative) directly costs the firm money, which is worse than occasionally flagging something as extra work that wasn't (a false positive, which a human would just double check).

## 7. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, lr_preds, labels=lr.classes_)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=lr.classes_, yticklabels=lr.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Logistic Regression')
plt.tight_layout()
plt.show()

## 8. Testing on Genuinely New, Unseen Messages

The real test of a model isn't its accuracy on the test split — it's how it handles messages that look nothing like anything it was trained on. Below are three brand-new example sentences, written fresh, that were never part of the dataset at all.

In [ ]:
new_examples = [
    "Please have the final logo files ready by next Monday.",
    "Can you check the invoice numbers for last quarter?",
    "We'd like an additional landing page built — this wasn't in the original quote.",
]

new_vec = vectorizer.transform(new_examples)
predictions = lr.predict(new_vec)

for text, pred in zip(new_examples, predictions):
    print(f"[{pred:12s}] {text}")

## 9. Comparing This Approach to the LLM Pipeline

| | Trained classifier (this notebook) | LLM extraction (used in the live app) |
|---|---|---|
| Setup effort | Requires a labeled dataset and training step | Works immediately with a well-written prompt, no training data needed |
| Speed | Instant, runs locally, no network call | Depends on an external API call (typically under a second, but adds latency and a point of failure) |
| Cost per request | Free after training | Each request costs a small amount / counts against API quota |
| Extracts summary, due date, value | No — this model only classifies the type | Yes — a single prompt returns all fields at once |
| Handles messages very different from training data | Only as good as the training examples given | Generalises much better, since it's already been trained on a vast range of language |
| Improves over time | Only if retrained on new labeled data | Improves automatically as the underlying model is upgraded by the provider |

**Conclusion:** the trained classifier is a fast, free, and fully offline way to double-check the *type* classification specifically, but it cannot replace the LLM's ability to also extract a summary, a due date, and an estimated value in one step, nor does it generalise as well to messages very different from its training examples. In a production version of TrackItRight, a trained classifier like this could realistically be used as a **fast first-pass filter or a confidence check**, falling back to the LLM for anything the classifier is unsure about — rather than fully replacing it.

## 10. Saving the Trained Model

In [ ]:
joblib.dump(lr, 'model.joblib')
joblib.dump(vectorizer, 'vectorizer.joblib')

print("Saved model.joblib and vectorizer.joblib")

## 11. Loading and Reusing the Saved Model

This is how the saved model could be loaded later — for example, inside a small Python service that the main Node.js backend calls.

In [ ]:
loaded_model = joblib.load('model.joblib')
loaded_vectorizer = joblib.load('vectorizer.joblib')

sample = ["Client wants an extra feature added to the dashboard, please quote it."]
sample_vec = loaded_vectorizer.transform(sample)
print(loaded_model.predict(sample_vec))